# 02 — Rule-Based Routing: Examples

> Build a fully functional rule engine that routes requests to the right LLM based on context, cost, and latency rules.

---

**What you'll build:**
1. Token-count router
2. Keyword-based router
3. Cost-ceiling router
4. Latency-budget router
5. Composable `RuleEngine` class
6. Production rule engine with logging

## Setup

In [ ]:
# Install dependencies
# !pip install tiktoken openai python-dotenv rich

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Callable, Optional
import tiktoken
from rich import print as rprint
from rich.table import Table
from rich.console import Console

console = Console()
enc = tiktoken.encoding_for_model("gpt-4o-mini")

print('✅ Setup complete')

---
## Example 1: Token-Count Router

Route based on how many tokens the input contains.

In [ ]:
def count_tokens(messages: list[dict]) -> int:
    """Count total tokens in a message list."""
    return sum(len(enc.encode(m.get("content", ""))) for m in messages)


def route_by_token_count(messages: list[dict]) -> tuple[str, str]:
    """
    Route to a model based on total token count.
    Returns (model_name, reason).
    """
    tokens = count_tokens(messages)

    if tokens > 800_000:
        return "gemini-1.5-pro", f"{tokens:,} tokens → needs 2M context window"
    elif tokens > 100_000:
        return "claude-3-5-sonnet-20241022", f"{tokens:,} tokens → needs 200k context window"
    elif tokens > 64_000:
        return "gpt-4o", f"{tokens:,} tokens → needs 128k context window"
    else:
        return "gpt-4o-mini", f"{tokens:,} tokens → fits in 16k context (mini model)"


# ── Test with different message sizes ──────────────────────────────────────
test_cases = [
    [{"role": "user", "content": "What is 2 + 2?"}],
    [{"role": "user", "content": "Explain quantum computing. " * 3000}],
    [{"role": "user", "content": "Analyze this document: " + "word " * 70_000}],
]

table = Table(title="Token-Count Router", show_header=True)
table.add_column("Tokens", style="cyan")
table.add_column("Selected Model", style="green")
table.add_column("Reason", style="white")

for msgs in test_cases:
    model, reason = route_by_token_count(msgs)
    tokens = count_tokens(msgs)
    table.add_row(f"{tokens:,}", model, reason)

console.print(table)

---
## Example 2: Keyword-Based Router

In [ ]:
# Define keyword sets per task domain
ROUTING_KEYWORDS = {
    "claude-3-5-sonnet-20241022": [
        "python", "javascript", "typescript", "def ", "class ", "function",
        "debug", "bug", "code", "algorithm", "implement", "refactor", "optimize"
    ],
    "o1-mini": [
        "calculate", "equation", "integral", "derivative", "proof",
        "solve for", "theorem", "matrix", "probability"
    ],
    "claude-3-5-sonnet-20241022_creative": [
        "write a story", "poem", "creative", "imagine", "fiction", "narrative", "novel"
    ],
}

# Flatten for routing lookup
KEYWORD_MODEL_MAP = [
    ("claude-3-5-sonnet-20241022",  ["python", "javascript", "def ", "class ", "function", "debug", "code", "algorithm", "implement"]),
    ("o1-mini",                     ["calculate", "equation", "integral", "derivative", "proof", "solve for", "probability"]),
    ("claude-3-5-sonnet-20241022",  ["write a story", "poem", "creative", "fiction", "narrative"]),
]

def route_by_keywords(text: str, default: str = "gpt-4o-mini") -> tuple[str, str]:
    """Route by detecting task-type keywords in the user text."""
    text_lower = text.lower()
    for model, keywords in KEYWORD_MODEL_MAP:
        matched = [kw for kw in keywords if kw in text_lower]
        if matched:
            return model, f"Matched keywords: {matched}"
    return default, "No keywords matched — using default model"


# ── Test ───────────────────────────────────────────────────────────────────
test_prompts = [
    "Write a Python function to sort a list of dictionaries by a key",
    "Calculate the integral of x^2 dx from 0 to 1",
    "Write a short creative story about a robot learning to paint",
    "What is the capital of France?",
    "Implement a binary search algorithm in JavaScript",
]

table = Table(title="Keyword Router", show_header=True)
table.add_column("Prompt", style="cyan", max_width=50)
table.add_column("Model", style="green")
table.add_column("Reason", style="white")

for prompt in test_prompts:
    model, reason = route_by_keywords(prompt)
    table.add_row(prompt[:48] + ".." if len(prompt) > 48 else prompt, model, reason)

console.print(table)

---
## Example 3: Cost-Ceiling Router

In [ ]:
# Pricing: (input_per_token, output_per_token) in USD
PRICING = {
    "gpt-4o-mini":              (0.15e-6,  0.60e-6),
    "gemini-1.5-flash":         (0.075e-6, 0.30e-6),
    "claude-3-5-haiku-20241022":(0.80e-6,  4.00e-6),
    "gpt-4o":                   (5.00e-6, 15.00e-6),
    "claude-3-5-sonnet-20241022":(3.00e-6, 15.00e-6),
    "o1":                       (15.0e-6, 60.00e-6),
}

# Quality ranking (best → cheapest)
QUALITY_ORDER = ["o1", "claude-3-5-sonnet-20241022", "gpt-4o", "claude-3-5-haiku-20241022", "gpt-4o-mini", "gemini-1.5-flash"]


def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    pi, po = PRICING[model]
    return pi * input_tokens + po * output_tokens


def route_by_budget(
    input_tokens: int,
    output_tokens: int,
    max_cost_usd: float
) -> tuple[str, str]:
    """Select highest-quality model that fits within the cost ceiling."""
    for model in QUALITY_ORDER:
        cost = estimate_cost(model, input_tokens, output_tokens)
        if cost <= max_cost_usd:
            return model, f"Cost ${cost:.6f} ≤ ceiling ${max_cost_usd:.4f}"
    fallback = "gemini-1.5-flash"
    return fallback, "All models exceed budget — using cheapest model"


# ── Test ───────────────────────────────────────────────────────────────────
scenarios = [
    (500,  200,  0.001,  "Tight budget"),
    (2000, 1000, 0.01,   "Medium budget"),
    (5000, 2000, 0.10,   "Generous budget"),
    (1000, 500,  0.0001, "Ultra tight budget"),
]

table = Table(title="Cost-Ceiling Router", show_header=True)
table.add_column("Scenario", style="cyan")
table.add_column("Budget", style="yellow")
table.add_column("Selected Model", style="green")
table.add_column("Estimated Cost", style="white")

for in_tok, out_tok, budget, label in scenarios:
    model, reason = route_by_budget(in_tok, out_tok, budget)
    cost = estimate_cost(model, in_tok, out_tok)
    table.add_row(label, f"${budget:.4f}", model, f"${cost:.6f}")

console.print(table)

---
## Example 4: Latency-Budget Router

In [ ]:
# Approximate p50 latency (ms) per model from public benchmarks
MODEL_LATENCY_MS = {
    "gemini-2.0-flash":          500,
    "gemini-1.5-flash":          600,
    "claude-3-5-haiku-20241022": 800,
    "gpt-4o-mini":              1200,
    "gpt-4o":                   2000,
    "claude-3-5-sonnet-20241022":3000,
    "o1-mini":                  8000,
    "o1":                      20000,
}

# Sort models by quality (simple rank for demo)
QUALITY_RANK = {
    "o1": 10, "claude-3-5-sonnet-20241022": 9, "gpt-4o": 8,
    "o1-mini": 7, "claude-3-5-haiku-20241022": 6, "gpt-4o-mini": 5,
    "gemini-1.5-flash": 4, "gemini-2.0-flash": 3
}

def route_by_latency(max_latency_ms: int) -> tuple[str, str]:
    """Select highest-quality model that meets the latency budget."""
    eligible = {m: l for m, l in MODEL_LATENCY_MS.items() if l <= max_latency_ms}
    if not eligible:
        fastest = min(MODEL_LATENCY_MS, key=MODEL_LATENCY_MS.get)
        return fastest, f"No model meets {max_latency_ms}ms — using fastest: {MODEL_LATENCY_MS[fastest]}ms"
    # Among eligible, pick highest quality
    best = max(eligible, key=lambda m: QUALITY_RANK.get(m, 0))
    return best, f"Latency {MODEL_LATENCY_MS[best]}ms ≤ {max_latency_ms}ms budget"


# ── Test ───────────────────────────────────────────────────────────────────
latency_budgets = [400, 800, 1500, 3000, 25000]

table = Table(title="Latency Router", show_header=True)
table.add_column("Max Latency", style="cyan")
table.add_column("Selected Model", style="green")
table.add_column("Reason", style="white")

for budget in latency_budgets:
    model, reason = route_by_latency(budget)
    table.add_row(f"{budget}ms", model, reason)

console.print(table)

---
## Example 5: Composable RuleEngine

In [ ]:
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class RoutingRule:
    name: str
    condition: Callable[[dict], bool]
    model: str
    priority: int  # Lower = evaluated first
    reason: str


class RuleEngine:
    """
    Priority-ordered rule engine for LLM routing.
    First matching rule wins. Falls back to default_model.
    """

    def __init__(self, default_model: str = "gpt-4o-mini"):
        self.rules: list[RoutingRule] = []
        self.default = default_model

    def add_rule(self, rule: RoutingRule) -> "RuleEngine":
        self.rules.append(rule)
        self.rules.sort(key=lambda r: r.priority)
        return self  # Enable chaining

    def route(self, context: dict) -> tuple[str, str]:
        """Evaluate rules and return (model, reason)."""
        for rule in self.rules:
            try:
                if rule.condition(context):
                    return rule.model, f"[Rule '{rule.name}'] {rule.reason}"
            except Exception as e:
                print(f"⚠️ Rule '{rule.name}' error: {e} — skipping")
        return self.default, "No rules matched — using default model"

    def explain(self, context: dict):
        """Show which rules passed/failed for a given context."""
        table = Table(title="Rule Evaluation Trace", show_header=True)
        table.add_column("Priority", style="cyan")
        table.add_column("Rule", style="white")
        table.add_column("Matched?", style="yellow")
        table.add_column("Model", style="green")

        for rule in self.rules:
            try:
                matched = rule.condition(context)
            except:
                matched = False
            table.add_row(
                str(rule.priority),
                rule.name,
                "✅ YES" if matched else "❌ no",
                rule.model if matched else "-"
            )
        console.print(table)


print("✅ RuleEngine defined")

In [ ]:
# ── Build a production-style rule engine ──────────────────────────────────

engine = RuleEngine(default_model="gpt-4o-mini")

(
    engine
    .add_rule(RoutingRule(
        name="long_context",
        condition=lambda ctx: ctx.get("input_tokens", 0) > 100_000,
        model="claude-3-5-sonnet-20241022",
        priority=1,
        reason="Input > 100k tokens → needs 200k context model"
    ))
    .add_rule(RoutingRule(
        name="realtime",
        condition=lambda ctx: ctx.get("max_latency_ms", float("inf")) < 1000,
        model="claude-3-5-haiku-20241022",
        priority=2,
        reason="Latency budget < 1s → fastest model"
    ))
    .add_rule(RoutingRule(
        name="coding_task",
        condition=lambda ctx: any(
            kw in ctx.get("text", "").lower()
            for kw in ["def ", "class ", "function", "debug", "code", "algorithm", "implement"]
        ),
        model="claude-3-5-sonnet-20241022",
        priority=3,
        reason="Coding keywords detected"
    ))
    .add_rule(RoutingRule(
        name="math_reasoning",
        condition=lambda ctx: any(
            kw in ctx.get("text", "").lower()
            for kw in ["calculate", "equation", "proof", "solve for", "integral", "probability"]
        ),
        model="o1-mini",
        priority=4,
        reason="Math/reasoning task detected"
    ))
    .add_rule(RoutingRule(
        name="tight_budget",
        condition=lambda ctx: ctx.get("max_cost_usd", float("inf")) < 0.001,
        model="gemini-1.5-flash",
        priority=5,
        reason="Budget ceiling < $0.001/call → cheapest model"
    ))
)

print("✅ Rule engine configured with 5 rules")

In [ ]:
# ── Test the rule engine ───────────────────────────────────────────────────

test_contexts = [
    {"text": "Write a Python class to parse JSON", "input_tokens": 50},
    {"text": "What is 2 + 2?", "max_latency_ms": 500},
    {"text": "Summarize this 200-page document", "input_tokens": 150_000},
    {"text": "Calculate the integral of sin(x) from 0 to pi"},
    {"text": "What is the weather like today?", "max_cost_usd": 0.0005},
    {"text": "What is the meaning of life?"},
]

table = Table(title="Rule Engine Results", show_header=True)
table.add_column("Request", style="cyan", max_width=45)
table.add_column("Model", style="green")
table.add_column("Matched Rule", style="white")

for ctx in test_contexts:
    model, reason = engine.route(ctx)
    text = ctx.get("text", "")[:43] + ".." if len(ctx.get("text", "")) > 43 else ctx.get("text", "")
    table.add_row(text, model, reason)

console.print(table)

In [ ]:
# ── Rule trace — explain routing decision ──────────────────────────────────

context = {"text": "Implement a binary search tree in Python", "input_tokens": 200}
print(f"🔍 Explaining routing for: \"{context['text']}\"\n")
engine.explain(context)

model, reason = engine.route(context)
rprint(f"\n[bold green]→ Final decision: {model}[/bold green]")
rprint(f"   Reason: {reason}")

---
## Example 6: Rule Engine with Live LLM Call

In [ ]:
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def routed_completion(user_message: str, **constraints) -> dict:
    """
    Route a request through the rule engine and call the selected model.
    constraints: max_latency_ms, max_cost_usd, input_tokens, etc.
    """
    tokens = count_tokens([{"role": "user", "content": user_message}])
    context = {"text": user_message, "input_tokens": tokens, **constraints}

    model, reason = engine.route(context)

    # Map to OpenAI-compatible model name if needed
    openai_models = {"gpt-4o-mini", "gpt-4o", "o1-mini", "o1"}
    call_model = model if model in openai_models else "gpt-4o-mini"  # Fallback for demo

    print(f"\n📤 Request: {user_message[:60]}...")
    print(f"   → Routed to: {model} ({reason})")
    if call_model != model:
        print(f"   → Demo: calling gpt-4o-mini instead of {model}")

    start = time.time()
    response = client.chat.completions.create(
        model=call_model,
        messages=[{"role": "user", "content": user_message}],
        max_tokens=150
    )
    latency = (time.time() - start) * 1000

    return {
        "content": response.choices[0].message.content,
        "model_routed_to": model,
        "model_called": call_model,
        "routing_reason": reason,
        "latency_ms": round(latency, 1),
        "tokens_used": response.usage.total_tokens,
    }


# Run a few routed calls
prompts = [
    ("Implement a quicksort algorithm in Python", {}),
    ("What is the capital of Japan?", {"max_latency_ms": 500}),
    ("Calculate the probability of rolling a 6 three times in a row", {}),
]

results = []
for prompt, constraints in prompts:
    result = routed_completion(prompt, **constraints)
    results.append(result)
    print(f"   ✅ Response ({result['latency_ms']}ms): {result['content'][:80]}...\n")

---
## Summary

| Component | What It Does |
|---|---|
| `route_by_token_count()` | Routes based on input size | 
| `route_by_keywords()` | Routes based on task type keywords |
| `route_by_budget()` | Selects best model within cost ceiling |
| `route_by_latency()` | Selects best model meeting latency SLA |
| `RuleEngine` | Composable, priority-ordered rule evaluator |
| `routed_completion()` | End-to-end routed LLM call |

**Next:** `03_semantic_routing` — route based on *meaning*, not just keywords.